## 1. OpenAI Function Calling의 개념

### 텍스트 기반 ReAct vs Function Calling

이전에 구현한 텍스트 기반 ReAct 에이전트는 LLM의 출력을 직접 파싱하여 어떤 도구를 호출할지 결정했다. 이 방식은 유연하지만, 문자열 파싱에 의존하기 때문에 오류가 발생할 가능성이 높다.

OpenAI의 **Function Calling**은 이 문제를 구조적으로 해결한다. LLM이 자유 텍스트 대신 **구조화된 JSON 형태**로 함수 호출 정보를 반환하므로, 별도의 파싱 로직 없이도 안정적으로 도구를 사용할 수 있다.

### Function Calling의 주요 장점

| 장점 | 설명 |
|------|------|
| 구조화된 출력 | JSON 형태로 함수명과 인자를 반환하여 파싱 오류를 방지 |
| 타입 안전성 | JSON Schema로 인자의 타입과 제약조건을 정의 가능 |
| 병렬 호출 | 여러 함수를 동시에 호출할 수 있는 Parallel Function Calling 지원 |
| 선택적 제어 | tool_choice 파라미터로 도구 호출 방식을 세밀하게 제어 |

### tools 파라미터와 JSON Schema

Function Calling을 사용하려면 API 호출 시 `tools` 파라미터에 사용 가능한 함수들의 스키마를 전달한다. 각 함수는 다음과 같은 JSON Schema 형식으로 정의된다:

```python
tools = [{
    "type": "function",
    "function": {
        "name": "함수명",
        "description": "함수 설명",
        "parameters": {
            "type": "object",
            "properties": {
                "param1": {"type": "string", "description": "설명"}
            },
            "required": ["param1"]
        }
    }
}]
```

### tool_choice 옵션

| 옵션 | 동작 |
|------|------|
| `"auto"` | 모델이 자동으로 함수 호출 여부를 결정 (기본값) |
| `"required"` | 반드시 하나 이상의 함수를 호출하도록 강제 |
| `"none"` | 함수 호출을 하지 않고 텍스트만 생성 |
| `{"type": "function", "function": {"name": "함수명"}}` | 특정 함수를 반드시 호출 |

In [10]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY') )
# JSON Schema를 사용하여 도구 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름 (예: Seoul, Tokyo)"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "온도 단위"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "수학 계산을 수행합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 수식 (예: 2 + 3 * 4)"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]
# 도구 구현 함수
from numpy import sqrt

import requests
def get_weather(city,unit='celsius'):
    '''wttr.in 공개 api를 사용해서 실제 날씨를 조회'''
    try:
        url = f'https://wttr.in/{city}?format=j1'
        resp = requests.get(url,timeout=10)
        data = resp.json()
        current = data['current_condition'][0]
        temp_c = current['temp_C']
        temp_f = current['temp_F']
        desc = current['weatherDesc'][0]['value']
        humidity = current['humidity']
        temp = temp_c if unit=='celsius' else temp_f
        unit_str = 'C' if unit=='celsius' else 'F'
        return json.dumps({
            'city':city,
            'temperature' : f'{temp} degreees {unit_str}',
            'description' : desc,
            'humidity' : f'{humidity}%'
        },ensure_ascii=False)
    except Exception as e:
        return json.dumps({'city':city, 'temperature' : '22 degrees C'})

def calculate(expression):
    """수학 수식을 안전하게 계산"""
    try:
        result = eval(expression)
        return json.dumps({"expression": expression, "result": str(result)})
    except Exception as e:
        return json.dumps({"error": str(e)})
# 함수명으로 실제 함수를 매핑하는 dictionary  이전 코드에서 tools 역활
available_functions = {
    'get_weather':get_weather,
    'calculate' : calculate
}         

# Function calling을 포함한 api호출
response = client.chat.completions.create(
    model='gpt-5.4-nano',
    messages=[{'role':'user','content':'root(((2026-2020)33 -20)**2)'}],
    tools=tools,
    tool_choice='auto',
    temperature=0
)
print('======= Funtion Calling 응답 분석 =========')
message = response.choices[0].message
print(f'Content : {message.content}')
print(f'Tool Calls : {message.tool_calls}')
if message.tool_calls:
    for tool_call in message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        print(f'\n호출된 함수 : {func_name}')
        print(f'인자 : {func_args}')
        # 함수실행
        result = available_functions[func_name](**func_args)
        print(f'실행 결과 : {result}')

======= Funtion Calling 응답 분석 =========
Content : None
Tool Calls : [ChatCompletionMessageFunctionToolCall(id='call_30STt3TR0lYRDYCyCMpbPjsX', function=Function(arguments='{"expression":"sqrt(((2026-2020)*33 -20)^2)"}', name='calculate'), type='function')]

호출된 함수 : calculate
인자 : {'expression': 'sqrt(((2026-2020)*33 -20)^2)'}
실행 결과 : {"expression": "sqrt(((2026-2020)*33 -20)^2)", "result": "13.2664991614216"}


## 3. Function Calling 기반 ReAct 루프

Function Calling을 사용한 ReAct 루프는 텍스트 기반 방식보다 훨씬 구조적이다. 전체 흐름은 다음과 같다:

1. 사용자 질문을 메시지에 추가하고 API 호출
2. 응답에 `tool_calls`가 있으면 해당 함수를 실행
3. 실행 결과를 `role: tool` 메시지로 추가하고 다시 API 호출
4. `tool_calls`가 없으면 (최종 텍스트 응답) 루프 종료

이 과정에서 텍스트 파싱이 전혀 필요하지 않다는 것이 핵심적인 차이점이다.

In [11]:
def run_function_calling_agent(question, tools, available_functions, max_steps=5):
    """Function Calling 기반 ReAct 에이전트 루프"""
    messages = [{"role": "user", "content": question}]

    print(f"Question: {question}")
    print("=" * 60)

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-5.4-nano-2026-03-17",
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            print(f"\n--- Step {step + 1}: Tool Calls ---")
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                print(f"  Function: {func_name}({func_args})")

                result = available_functions[func_name](**func_args)
                print(f"  Result: {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        else:
            # tool_calls가 없으면 최종 응답
            print(f"\n--- Final Response ---")
            print(message.content)
            return message.content

    return "최대 단계 도달"

# 다단계 질문으로 테스트: 날씨 비교 + 온도 차이 계산
result = run_function_calling_agent(
    "서울과 도쿄의 현재 날씨를 비교해주세요. 두 도시의 온도 차이도 계산해주세요.",
    tools,
    available_functions
)

Question: 서울과 도쿄의 현재 날씨를 비교해주세요. 두 도시의 온도 차이도 계산해주세요.

--- Step 1: Tool Calls ---
  Function: get_weather({'city': 'Seoul', 'unit': 'celsius'})
  Result: {"city": "Seoul", "temperature": "25 degreees C", "description": "Partly cloudy", "humidity": "79%"}
  Function: get_weather({'city': 'Tokyo', 'unit': 'celsius'})
  Result: {"city": "Tokyo", "temperature": "22 degreees C", "description": "Partly cloudy", "humidity": "100%"}

--- Step 2: Tool Calls ---
  Function: calculate({'expression': '25-22'})
  Result: {"expression": "25-22", "result": "3"}

--- Final Response ---
현재 날씨 비교(섭씨 기준):

- **서울**: **25°C**, 부분적으로 흐림, 습도 **79%**
- **도쿄**: **22°C**, 부분적으로 흐림, 습도 **100%**

**온도 차이**: **서울 25°C - 도쿄 22°C = 3°C** (서울이 **3°C 더 높음**)


## 4. Parallel Function Calling

Parallel Function Calling은 모델이 단일 턴에서 여러 개의 함수를 동시에 호출할 수 있는 기능이다. 바로 이 기능의 예시이다.

### 단일 호출 vs 병렬 호출

| 방식 | 동작 | 효율성 |
|------|------|--------|
| 단일 호출 | 한 번에 하나의 함수만 호출, 결과 확인 후 다음 호출 | API 호출 횟수 증가 |
| 병렬 호출 | 독립적인 여러 함수를 한 턴에서 동시 호출 | API 호출 횟수 감소, 지연 시간 단축 |

모델은 여러 함수 호출이 서로 독립적일 때 자동으로 병렬 호출을 수행한다. 예를 들어 세 도시의 날씨를 동시에 조회하는 경우가 해당된다.